# 🧠 Module 3: Spatial-Temporal Feature Extraction via Deep Autoencoders

## 📌 1. Overview & Motivation
In 6G Dynamic Spectrum Sharing, multiple RF channels ($C=16$) generate complex, high-dimensional multi-sensor power spectral observations at every time step $t$.

This notebook trains a **Deep Bottleneck Autoencoder** to compress multi-channel spatial-temporal spectrum snapshot vectors into a compact, low-dimensional feature embedding $h_t$. These bottleneck features preserve critical inter-channel spatial correlations while filtering high-frequency thermal noise.

---

## 📐 2. Mathematical Formulation of Autoencoder Bottleneck

### A. Spatial Snapshot Vector Representation
At time step $t$, the full spectrum snapshot vector $X_t \in \mathbb{R}^{D}$ combines power, SNR, noise, and interference across all 16 RF channels ($D = 16 \times 4 = 64$ features).

### B. Deep Autoencoder Architecture
1. **Encoder Network** $g_\phi: \mathbb{R}^D \to \mathbb{R}^k$:
   $$h_t = \text{ReLU}\left(W_2 \cdot \text{ReLU}\left(W_1 X_t + b_1\right) + b_2\right)$$
   where $h_t \in \mathbb{R}^k$ ($k=8$) is the bottleneck representation ($8\times$ dimensionality compression).

2. **Decoder Network** $f_\theta: \mathbb{R}^k \to \mathbb{R}^D$:
   $$\hat{X}_t = W_4 \cdot \text{ReLU}\left(W_3 h_t + b_3\right) + b_4$$

### C. Objective Function (Mean Squared Error - MSE)
The network parameters $(\theta, \phi)$ are optimized by minimizing input-reconstruction discrepancy:
$$\mathcal{L}_{AE}(\theta, \phi) = \frac{1}{N} \sum_{i=1}^N \| X^{(i)} - \hat{X}^{(i)} \|_2^2$$

---

## 🛠️ 3. Snapshot Data Preparation

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn

# Ensure project root is in Python path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dss.data_generation.simulator import generate_dataset
from dss.feature_extraction.snapshots import build_snapshots_from_df
from dss.feature_extraction.autoencoder import Autoencoder
from dss.feature_extraction.train import train_autoencoder

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device configured: {device}")

## 🔄 4. Snapshot Vector Assembly & Scaling
We assemble snapshot matrices $X \in \mathbb{R}^{T \times 64}$ where $T=4032$ time steps and each row contains 64 multi-channel features.

In [ ]:
# Generate dataset
channel_df, _, _ = generate_dataset("normal", seed=42, save=False)

# Build multi-channel snapshot vectors
snapshots, ts_list = build_snapshots_from_df(channel_df)
print(f"Snapshot Tensor Shape: {snapshots.shape} (Time Steps x Features per step)")

# Split into Train (80%) and Test (20%)
split_idx = int(len(snapshots) * 0.8)
X_train = snapshots[:split_idx]
X_test = snapshots[split_idx:]

print(f"Train snapshots: {X_train.shape}, Test snapshots: {X_test.shape}")

## 🧱 5. Autoencoder Neural Network Training
We initialize our PyTorch `Autoencoder` (Compressing 64 dimensions to 8 bottleneck dimensions) and train for 30 epochs.

In [ ]:
input_dim = snapshots.shape[1]  # 64
bottleneck_dim = 8            # 8 latent bottleneck features

ae_model = Autoencoder(input_dim=input_dim, latent_dim=bottleneck_dim).to(device)

# Train Autoencoder
ae_model, history = train_autoencoder(
    model=ae_model,
    X_train=X_train,
    X_val=X_test,
    epochs=30,
    batch_size=128,
    lr=1e-3,
    device=str(device)
)

# Plot MSE Reconstruction Loss
plt.figure(figsize=(10, 4))
plt.plot(history["train_loss"], label="Train MSE Loss", linewidth=2)
plt.plot(history["val_loss"], label="Test MSE Loss", linewidth=2, linestyle="--")
plt.title("Deep Autoencoder Snapshot Reconstruction Loss", fontsize=13, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Reconstruction MSE Loss")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

## 🔍 6. Visualizing Snapshot Reconstruction Performance
We compare original input spectrum snapshots against autoencoder reconstructed snapshots to verify feature preservation.

In [ ]:
ae_model.eval()
with torch.no_grad():
    sample_input = torch.tensor(X_test[:50], dtype=torch.float32).to(device)
    encoded_features = ae_model.encode(sample_input)
    _, reconstructed = ae_model(sample_input)
    reconstructed = reconstructed.cpu().numpy()

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

sns.heatmap(X_test[:50].T, ax=axes[0], cmap="viridis", cbar_kws={'label': 'Normalized Value'})
axes[0].set_title("Original Spectrum Snapshot Vector (50 Time Steps x 64 Features)", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Feature Dim")

sns.heatmap(reconstructed.T, ax=axes[1], cmap="viridis", cbar_kws={'label': 'Reconstructed Value'})
axes[1].set_title("Autoencoder Reconstructed Snapshot Vector", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Time Step Index")
axes[1].set_ylabel("Feature Dim")

plt.tight_layout()
plt.show()

## 🧪 7. Bottleneck Feature Extraction & Embedding Export
Finally, we pass all spectrum snapshots through the trained encoder $g_\phi$ to extract the compact 8-dimensional bottleneck features.

In [ ]:
with torch.no_grad():
    full_input = torch.tensor(snapshots, dtype=torch.float32).to(device)
    latent_embeddings = ae_model.encode(full_input).cpu().numpy()

print(f"🎉 Feature Extraction Complete!")
print(f"Extracted Latent Embeddings Shape: {latent_embeddings.shape} (T x 8 features)")

# Display latent bottleneck feature values for first 5 time steps
latent_df = pd.DataFrame(latent_embeddings, columns=[f"latent_feat_{i+1}" for i in range(8)])
latent_df.head()

## 🎯 Summary & Conclusion
- Spatial-Temporal Snapshot Autoencoder trained end-to-end.
- 64-dimensional multi-channel RF feature vectors compressed into 8 latent bottleneck features ($8\times$ compression ratio).
- Low MSE reconstruction loss confirms high-fidelity feature representation.
- **Workflow Complete!** All 3 modules (`Data Generation & DB`, `VAE-GMM Sensing`, and `Autoencoder Feature Extraction`) successfully implemented in Jupyter Notebooks.